# HR.TRG_LOC Conversion Notebook

**Source File:** `TARGET.sql`
**Conversion Date:** 2024-07-30
**Description:** Direct insert into `TRG_LOC` from `LOCATIONS` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT current_timestamp() AS etl_current_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('1900-01-01', 'yyyy-MM-dd') AS etl_last_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id_param;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid_param;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no_param;

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS etl_job_type,
    vce.etl_current_extract_time AS etl_current_extract_time,
    vle.etl_last_extract_time AS etl_last_extract_time,
    vd.datasource_num_id_param AS datasource_num_id,
    vp.etl_proc_wid_param AS etl_proc_wid,
    vo.odi_sess_no_param AS odi_sess_no
  FROM
    v_etl_current_extract_time vce,
    v_etl_last_extract_time vle,
    v_datasource_num_id vd,
    v_etl_proc_wid vp,
    v_odi_sess_no vo
"""))

# Staging Table

# Flow Table

# Data Load

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}
INSERT INTO workspace.hr.trg_loc
  (
    location_id,
    street_address,
    postal_code,
    city,
    state_province,
    country_id
  )
SELECT
  locations.location_id,
  locations.street_address,
  locations.postal_code,
  locations.city,
  locations.state_province,
  locations.country_id
FROM
  workspace.hr.locations AS locations;

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_loc;

# Cleanup

# Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

# Conversion Notes

- The original SQL was a simple direct insert statement from `HR.LOCATIONS` to `HR.TRG_LOC`.
- Oracle hints `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Databricks Delta Lake.
- Schema names have been converted to `workspace.hr` and table names to lowercase (`trg_loc`, `locations`).
- Column names in the INSERT and SELECT lists have been converted to lowercase for consistency.
- Standard ETL parameter widgets and temporary views have been included as per the notebook structure requirements, even though they are not directly referenced in this specific SQL statement.